# **Validation Agent**

Checks one SAS unit translated by the Translator (`notebooks/translator-*.ipynb`) and decides where it goes next
(Validation Agent Brief v1):

- **PASSED**: the translation follows the project rules and gives the same outputs as SAS.
- **TRIAGED**: it goes to the **Correction Agent** with a `fix_request` that says what failed.

The validator never edits code. Plain Python checks decide whether the code follows the rules and matches SAS.
Gemini then reads the code against the SAS source to find what the data can't show (hard-coded values, quiet
logic bugs) and to point at the line behind each failing check.

It is a LangGraph agent, like `sas_agent`. All functions and the graph are in **`validator.py`**; this
notebook only calls them.

```
START ─► static_check ─► execute ─► compare ─► review ─► verdict ─► END
              │             │ crash ─────────▲          ▲
              └─ rule broken ┴───────────────┴──────────┘  (code not run, review skipped)
```

| Node | State (item E) | Events | Uses the LLM? | What it does |
|---|---|---|---|---|
| `static_check` | TRANSLATED / PATCHED → CHECKED | `STATIC_PASS` / `STATIC_FAIL` | no | the five output fields; the code parses, has one function, safe imports, no file I/O, the declared checkpoints |
| `execute` | CHECKED → EXECUTED | `EXEC_OK` / `EXEC_ERROR` | no | calls the function on the test inputs |
| `compare` | EXECUTED → COMPARED | `COMPARED_DONE` | no | every expected output vs the SAS output |
| `review` | (part of COMPARED) | `REVIEW_DONE` | Gemini | hard-coded values, quiet bugs, the line behind each failing check |
| `verdict` | → PASSED / TRIAGED | `ALL_PASS` / `FAIL` | no | the next state, the rank and the fix request |

# **Setup**

The Gemini key is read from **`validation_agent/.env`**, never from the notebook. Put one line in that file:

```
GEMINI_API_KEY=your-key
```

(`GOOGLE_API_KEY=...` works too; `.env` is git-ignored.) Only the `review` node uses it.
Set `REVIEW = False` to run the manual checks only, with no API calls (`llm_review: "skipped"`).

In [ ]:
import json
from pathlib import Path

import pandas as pd

from validator import (
    MODEL,
    TRANSLATION_RESULT_FIELDS,
    build_fix_request,
    build_graph,
    build_llm,
    check_code,
    compare_outputs,
    llm_review,
    make_result,
    run_function,
    tool_findings,
    validate,
)

REVIEW = True                # False: manual checks only, no API calls

llm = build_llm() if REVIEW else None    # reads the key from validation_agent/.env
print("LLM review:", f"on ({MODEL})" if llm else "off")

## **Validator Input Contract**

The keys of the graph state. The first three are required:

- `translator_input`: the five inputs the Translator received: `specification`, `sas_source`,
  `upstream_signatures`, `behavior_notes`, `project_conventions`. The LLM review reads them.
- `translation`: the Translator's output, one object with exactly five fields:
    - `code`: the complete translated function.
    - `checkpoints`: the checkpoint names, in the order the code calls `checkpoint(name, df)`.
    - `conflicts`: places where the specification and the SAS source disagree.
    - `assumptions`: assumptions made by the Translator.
    - `helpers_requested`: helper functions the translation needs.
- `test_data`: what to run the function on.
    - `inputs`: the function's arguments by parameter name, e.g. `{"indata": DataFrame, "var": "..."}`.
    - `expected`: the SAS output for each name the function returns, e.g. `{"out": DataFrame}`.
- `program_id` (optional): the id the Correction Agent checks. Default: `specification.unit.id`.
- `attempt_dir` (optional): if given, the result is also written to `<attempt_dir>/ai/script.py` and
  `ai/validation.json`, the files the Correction Agent reads.

In [2]:
TRANSLATION_RESULT_FIELDS

('code', 'checkpoints', 'conflicts', 'assumptions', 'helpers_requested')

## **Validator Output Contract**

`ValidationResult` (graph state key `result`), the fields of Validation Agent Brief v1, item B:

| Field | Meaning |
|---|---|
| `program_id` | the unit or program checked; the Correction Agent checks it matches |
| `next_state` | `PASSED` or `TRIAGED` (state machine, item E) |
| `events` | one per stage, e.g. `STATIC_PASS`, `EXEC_OK`, `COMPARED_DONE`, `REVIEW_DONE`, `FAIL` |
| `stage_reached` | `static_check`, `execute` or `compare` |
| `rank` | `(-stage, failing checks, mismatches, largest error)`; lower is better (D-037) |
| `violations` | static-check rules broken |
| `exception` | `{type, line, source}` if the code crashed; never the message |
| `comparison` | redacted report: `passed`, `failing_checks`, `total_mismatches`, `max_error`, `checks` |
| `review` | LLM findings: `{kind, severity, python_lines, sas_lines, problem, what_to_check}` |
| `fix_request` | text for `{fix_request}` in `sas_agent/prompts/fix.md`; `None` when passed |

A comparison check is `{dataset, column, check, mismatch_count, max_error}`, where `check` is `file_missing`
(the output was not returned), `columns`, `row_count`, `row_order`, `missing_values` or `values`.
A review finding's `kind` is `hardcoded`, `quiet_bug` or `cause`. Without `REVIEW_DONE` in `events`, the review was off or skipped.

**PASSED** only if every output matches SAS, no review finding is `major`, and the Translator reported no
`conflicts` or `helpers_requested`.

# **Example: `%nation_avg` from Star_Macros.sas**

`%nation_avg` (Star_Macros.sas, lines 282-295) averages `grp_score` over the hospitals with at least 3
measures in a group. Program 2 calls it for each measure group; for mortality the result is
`Out_Mrt_Grp_Score_Nat` in `NATIONAL_AVERAGE_2025JUL`.

**Inputs**

In [3]:
DATA_DIR = Path("../data/Project_1")

macros = (DATA_DIR / "SAS Programs" / "Star_Macros.sas").read_text(encoding="utf-8", errors="replace").splitlines()
sas_source = "\n".join(macros[281:295])    # lines 282-295: %macro nation_avg ... %mend

translator_input = {
    "specification": {
        "unit": {"id": "M.nation_avg", "kind": "macro", "source": "Star_Macros.sas", "lines": [282, 295]},
        "interface": "nation_avg(indata: pd.DataFrame, var: str) -> dict[str, pd.DataFrame]",
        "returns": {"out": "one row, one column named `var`: the national mean"},
        "logic": [
            "keep hospitals with total_cnt >= 3",
            "take the mean of grp_score over those hospitals",
        ],
    },
    "sas_source": sas_source,
    "upstream_signatures": {},
    "behavior_notes": {"PROC SQL mean()": "ignores missing values; the mean of no values is missing"},
    "project_conventions": {
        "one_function_per_unit": True,
        "no_file_io_inside_units": True,
        "checkpoint_call": "checkpoint(name, dataframe) after every SAS dataset write",
        "allowed_imports": ["pandas", "numpy", "standard library"],
        "missing_values": "NaN",
    },
}

csv_dir = DATA_DIR / "SAS Output CSV"
test_data = {
    "inputs": {
        "indata": pd.read_csv(csv_dir / "OUTCOME_MORTALITY.csv", dtype={"PROVIDER_ID": str}),
        "var": "Out_Mrt_Grp_Score_Nat",
    },
    "expected": {
        "out": pd.read_csv(csv_dir / "NATIONAL_AVERAGE_2025JUL.csv")[["Out_Mrt_Grp_Score_Nat"]],
    },
}

print(sas_source)

%macro nation_avg(indata=, out=, var=);
   data temp;
	   set &indata;
	   where total_cnt>=3;**********;
	   keep provider_id total_cnt grp_score ;
	run;

	proc sql;
	   create table out1 as
	   select mean(grp_score) as &var from temp;
	quit;

	data &out(keep=&var);set out1;run;
%mend;


**Translations**

Written by hand in the Translator's output format. In the pipeline they come from the Translator
(e.g. `translate_with_reflection(...)` in the translator notebooks). The buggy one drops the
`where total_cnt>=3` filter (SAS line 4).

In [4]:
good_translation = {
    "code": '''import pandas as pd


def nation_avg(indata: pd.DataFrame, var: str) -> dict[str, pd.DataFrame]:
    # SAS L2-6: keep hospitals with at least 3 measures in the group
    temp = indata.loc[indata["total_cnt"] >= 3, ["PROVIDER_ID", "total_cnt", "grp_score"]]
    checkpoint("temp", temp)

    # SAS L8-11: PROC SQL mean() skips missing values, like pandas mean()
    out1 = pd.DataFrame({var: [temp["grp_score"].mean()]})
    checkpoint("out1", out1)

    # SAS L13: keep only the national mean
    out = out1[[var]]
    checkpoint("out", out)
    return {"out": out}
''',
    "checkpoints": ["temp", "out1", "out"],
    "conflicts": [],
    "assumptions": ["SAS provider_id is the PROVIDER_ID column (SAS names are case-insensitive)."],
    "helpers_requested": [],
}

buggy_translation = {
    **good_translation,
    "code": good_translation["code"].replace(
        'indata.loc[indata["total_cnt"] >= 3, ["PROVIDER_ID", "total_cnt", "grp_score"]]',
        'indata[["PROVIDER_ID", "total_cnt", "grp_score"]]',
    ),
}

# **Step by Step**

Each node of the graph calls one function from `validator.py`. Here they are called one at a time on the
buggy translation, to see what each step does.

## **Step 1: Static Checks** (`static_check` node)

Reads the code without running it. Returns the rules broken (`violations`); if there are any, the code is not run.

In [ ]:
program_id = translator_input["specification"]["unit"]["id"]
events = []

violations = check_code(buggy_translation)
events.append("STATIC_FAIL" if violations else "STATIC_PASS")

print(events[-1], violations)

## **Step 2: Execute the Function** (`execute` node)

Calls the function with `test_data["inputs"]`. `checkpoint(name, df)` does nothing here. A crash is
reported as `exception = {type, line, source}`: the error type and code line, not the message (which can contain data values).

In [ ]:
outputs, exception = run_function(buggy_translation["code"], test_data["inputs"])
events.append("EXEC_ERROR" if exception else "EXEC_OK")

print(events[-1], exception)
outputs["out"]

## **Step 3: Compare with SAS** (`compare` node)

For each expected output: it was returned, the columns and their order, the row count, the row order by
`PROVIDER_ID` (when the output has it), then every column (missing values, and values within `RTOL` / `ATOL`).
The result is the redacted report: names and counts only.

In [ ]:
comparison = compare_outputs(program_id, outputs, test_data["expected"])
events.append("COMPARED_DONE")

print(comparison.model_dump_json(indent=2))

## **Step 4: LLM Code Review** (`review` node)

Gemini (through LangChain) reads the SAS source, the numbered code and the tool findings (the same text
that starts the fix request). It never sees data values.

In [ ]:
findings_text = tool_findings(violations, exception, comparison)
print(findings_text, "\n")

review = llm_review(llm, translator_input, buggy_translation, findings_text) if llm else []
if llm:
    events.append("REVIEW_DONE")

for f in review:
    print(f"[{f.severity}] {f.kind}, Python {f.python_lines}, SAS {f.sas_lines}: {f.problem}")
print(len(review), "findings" if llm else "findings (LLM review is off)")

## **Step 5: Verdict and Fix Request** (`verdict` node)

**PASSED** only if every output matches SAS, Gemini found nothing `major` (`REVIEW_BLOCKS`), and the
Translator's `conflicts` and `helpers_requested` are empty. Otherwise **TRIAGED**, with the rank and a fix
request: the tool findings from the first failing stage, then the review findings, then any open issues.

In [ ]:
result = make_result(program_id, events, violations, exception, comparison, review, buggy_translation)

print(result.next_state, result.stage_reached, result.rank, result.events)
print()
print(result.fix_request)

# **The LangGraph Agent**

`build_graph(llm, review)` wires the same functions into nodes and edges, the same way as
`sas_agent.graph.build_graph`. Invoke it with the three inputs; the result is in `state["result"]`.

In [10]:
app = build_graph(llm, review=REVIEW)

state = app.invoke({
    "translator_input": translator_input,
    "translation": good_translation,
    "test_data": test_data,
})
print(state["result"].model_dump_json(indent=2))

{
  "status": "PASSED",
  "checks": [
    {
      "name": "output_format",
      "passed": true,
      "detail": "ok"
    },
    {
      "name": "syntax",
      "passed": true,
      "detail": "ok"
    },
    {
      "name": "one_function",
      "passed": true,
      "detail": "top-level functions: ['nation_avg']"
    },
    {
      "name": "imports",
      "passed": true,
      "detail": "ok"
    },
    {
      "name": "no_file_io",
      "passed": true,
      "detail": "ok"
    },
    {
      "name": "checkpoints",
      "passed": true,
      "detail": "declared ['temp', 'out1', 'out'], code calls ['temp', 'out1', 'out']"
    },
    {
      "name": "run",
      "passed": true,
      "detail": "returned ['out']"
    },
    {
      "name": "compare_out",
      "passed": true,
      "detail": "matches SAS"
    },
    {
      "name": "open_issues",
      "passed": true,
      "detail": "0 conflicts, helpers requested: []"
    }
  ],
  "review": [],
  "llm_review": "done",
  "fix_request

`validate(...)` does the same in one call and returns a dict.

In [ ]:
result = validate(translator_input, buggy_translation, test_data, llm, review=REVIEW)

print(result["next_state"], result["events"])
print(result["fix_request"])

# **Connecting with Other Agents**

Not wired yet: the Translator and the Correction Agent are still in progress. The wiring lives in
`validator.py`, not in this notebook:

- `build_graph()`: the validator, added to the pipeline as one node (a subgraph).
- `route_after_validation`: the edge after it. TRIAGED goes to `CORRECTION_NODE` (`"correct"`), PASSED goes to END.
- `correction_input(state)`: the arguments of `correction_agent.correct(program_id, code, validation)`.

Only the shared keys pass between the pipeline and the validator; its `events`, `violations`, ... stay inside it,
so every attempt starts fresh. Retry limits and escalation belong to the orchestrator.

```python
from correction_agent import correct
from validation_agent.validator import (
    CORRECTION_NODE, ValidationResult, build_graph as build_validator, correction_input, route_after_validation,
)

class PipelineState(TypedDict, total=False):
    program_id: str
    translator_input: dict
    translation: dict
    test_data: dict
    result: ValidationResult
    repair_prompt: str

def correct_node(state):
    return {"repair_prompt": correct(**correction_input(state)).repair_prompt}

pipeline = StateGraph(PipelineState)
pipeline.add_node("translate", translate)                  # Translator; uses repair_prompt on a retry
pipeline.add_node("validate", build_validator())
pipeline.add_node(CORRECTION_NODE, correct_node)
pipeline.add_edge(START, "translate")
pipeline.add_edge("translate", "validate")
pipeline.add_conditional_edges("validate", route_after_validation)
pipeline.add_edge(CORRECTION_NODE, "translate")
```